# Dataset Generator — Email A/B Test

Synthetic dataset generator for the email marketing A/B test analysis.
Produces a realistic, funnel-consistent dataset ready for analysis.

## What it does

Simulates 10,000 users receiving one of two email versions (control / treatment)
and records their behaviour at each stage of the email funnel.

The funnel is **causally enforced**:
- A user can only click if they opened
- A user can only convert if they clicked
- Unsubscribes are independent of funnel stage

## Parameters

```python
n_users = 10_000
np.random.seed(42)   # for reproducibility
```

| Event | Control | Treatment |
|-------|---------|-----------|
| Open rate | 25% | 30% |
| Click rate | 10% | 13% |
| Conversion rate | 10% | 12% |
| Unsubscribe rate | 2% | 4% |

Revenue is sampled from `Uniform(£20, £100)` for converters, 0 otherwise.

## Output

Saves `ab_test_email_campaign.csv` with the following columns:

| Column | Type | Description |
|--------|------|-------------|
| `user_id` | int | Sequential ID (1 → n_users) |
| `group` | str | `control` or `treatment` |
| `opened_email` | int (0/1) | Whether the user opened the email |
| `clicked_email` | int (0/1) | Whether the user clicked a link |
| `converted` | int (0/1) | Whether the user made a purchase |
| `unsubscribed` | int (0/1) | Whether the user unsubscribed |
| `revenue` | float | Purchase value in £ (0 if no purchase) |
| `timestamp` | datetime | One row per minute starting 2024-01-01 |

## Usage

```bash
pip install pandas numpy
jupyter notebook data_generator.ipynb
```

The CSV will be saved in the same directory and is ready to be loaded
directly by `AB_Testing_Business_Simple.ipynb`.

## Reproducibility

The seed `np.random.seed(42)` is set at the top of the notebook.
Re-running with the same seed produces the exact same dataset.
Change the seed or remove it entirely to generate a new random sample.

In [5]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_users = 10000

groups = np.random.choice(['control', 'treatment'], size=n_users)

# Probabilities
open_rate = {'control': 0.25, 'treatment': 0.30}
click_rate = {'control': 0.10, 'treatment': 0.13}
conversion_rate = {'control': 0.10, 'treatment': 0.12}
unsubscribe_rate = {'control': 0.02, 'treatment': 0.04}  # risk!

opened = []
clicked = []
converted = []
unsubscribed = []

for group in groups:
    o = np.random.rand() < open_rate[group]
    c = o and (np.random.rand() < click_rate[group])
    conv = c and (np.random.rand() < conversion_rate[group])
    unsub = np.random.rand() < unsubscribe_rate[group]

    opened.append(int(o))
    clicked.append(int(c))
    converted.append(int(conv))
    unsubscribed.append(int(unsub))

revenue = np.array(converted) * np.random.uniform(20, 100, size=n_users)

df = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    'group': groups,
    'opened_email': opened,
    'clicked_email': clicked,
    'converted': converted,
    'unsubscribed': unsubscribed,
    'revenue': revenue
})

df['timestamp'] = pd.date_range(start='2024-01-01', periods=n_users, freq='min')



df.head()

,user_id,group,opened_email,clicked_email,converted,unsubscribed,revenue,timestamp
0,1,control,0,0,0,0,0.0,2024-01-01 00:00:00
1,2,treatment,0,0,0,0,0.0,2024-01-01 00:01:00
2,3,control,0,0,0,0,0.0,2024-01-01 00:02:00
3,4,control,0,0,0,0,0.0,2024-01-01 00:03:00
4,5,control,1,0,0,0,0.0,2024-01-01 00:04:00


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   user_id        10000 non-null  int64  
 1   group          10000 non-null  object 
 2   opened_email   10000 non-null  int64  
 3   clicked_email  10000 non-null  int64  
 4   converted      10000 non-null  int64  
 5   unsubscribed   10000 non-null  int64  
 6   revenue        10000 non-null  float64
dtypes: float64(1), int64(5), object(1)
memory usage: 547.0+ KB


In [6]:
df.to_csv('ab_test_email_campaign.csv', index=False)